# 第12章

In [ ]:
!pip install pgmpy==0.1.24

## リスト12.1

In [ ]:
!pip install pgmpy==0.1.24
from pgmpy.models import BayesianNetwork
from pgmpy.factors.discrete import TabularCPD
from pgmpy.inference import VariableElimination
import numpy as np

model = BayesianNetwork([
    ('C', 'X'),
    ('C', 'Y'),
    ('X', 'Y'),
    ('Y', 'U')
])

## リスト12.2

In [ ]:
cpd_c = TabularCPD(
    variable='C',
    variable_card=2,
    values=[[0.5], [0.5]],
    state_names={'C': ['bear', 'bull']}
)

cpd_x = TabularCPD(
    variable='X',
    variable_card=2,
    values=[[0.8, 0.2], [0.2, 0.8]],
    evidence=['C'],
    evidence_card=[2],
    state_names={'X': ['debt', 'equity'], 'C': ['bear', 'bull']}
)

cpd_y = TabularCPD(
    variable='Y',
    variable_card=2,
    values= [[0.3, 0.9, 0.7, 0.6], [0.7, 0.1, 0.3, 0.4]],
    evidence=['X', 'C'],
    evidence_card=[2, 2],
    state_names={
        'Y': ['failure', 'success'],
        'X': ['debt', 'equity'],
        'C': ['bear', 'bull']
    }
)

## リスト12.3

In [ ]:
cpd_u = TabularCPD(
    variable='U',
    variable_card=2,
    values=[[1., 0.], [0., 1.]],
    evidence=['Y'],
    evidence_card=[2],
    state_names={'U': [-1000, 99000], 'Y': ['failure', 'success']}
)
print(cpd_u)
model.add_cpds(cpd_c, cpd_x, cpd_y, cpd_u)

## リスト12.4

In [ ]:
import requests

url = "https://raw.githubusercontent.com/altdeep/causalAI/master/book/pgmpy_do.py"
response = requests.get(url)
content = response.text

print("Downloaded script content:\n")
print(content)
confirm = input("\nDo you want to execute this script? (yes/no): ")
if confirm.lower() == 'yes':
    exec(content)
else:
    print("Script execution cancelled.")

## リスト12.5

In [ ]:
def get_expectation(marginal):
    u_values = marginal.state_names["U"]
    probs = marginal.values
    expectation = sum([x * p for x, p in zip(u_values, probs)])
    return expectation

infer = VariableElimination(model)
marginal_u_given_debt = infer.query(
    variables=['U'], evidence={'X': 'debt'})
marginal_u_given_equity = infer.query(
    variables=['U'], evidence={'X': 'equity'})
e_u_given_x_debt = get_expectation(marginal_u_given_debt)
e_u_given_x_equity = get_expectation(marginal_u_given_equity)
print("E(U(Y)|X=debt)=", e_u_given_x_debt)
print("E(U(Y)|X=equity)=", e_u_given_x_equity)

int_model_x_debt = do(model, {"X": "debt"})
infer_debt = VariableElimination(int_model_x_debt)
marginal_u_given_debt = infer_debt.query(variables=['U'])
expectation_u_given_debt = get_expectation(marginal_u_given_debt)
print("E(U(Y_{X=debt}))=", expectation_u_given_debt)
int_model_x_equity = do(model, {"X": "equity"})
infer_equity = VariableElimination(int_model_x_equity)
marginal_u_given_equity = infer_equity.query(variables=['U'])
expectation_u_given_equity = get_expectation(marginal_u_given_equity)
print("E(U(Y_{X=equity}))=", expectation_u_given_equity)

## リスト12.6

In [ ]:
model2 = BayesianNetwork([
    ('C', 'X'),
    ('C', 'Y'),
    ('X', 'Y'),
    ('Y', 'U')
])

cpd_y2 = TabularCPD(
    variable='Y',
    variable_card=2,
    values=[[0.3, 0.9, 0.7, 0.4],  [0.7, 0.1, 0.3, 0.6]],
    evidence=['X', 'C'],
    evidence_card=[2, 2],
    state_names={
        'Y': ['failure', 'success'],
        'X': ['debt', 'equity'],
        'C': ['bear', 'bull']
    }
)

model2.add_cpds(cpd_c, cpd_x, cpd_y2, cpd_u)

## リスト12.7

In [ ]:
infer = VariableElimination(model2)
marginal_u_given_debt = infer.query(
    variables=['U'], evidence={'X': 'debt'})
marginal_u_given_equity = infer.query(
    variables=['U'], evidence={'X': 'equity'})
e_u_given_x_debt = get_expectation(marginal_u_given_debt)
e_u_given_x_equity = get_expectation(marginal_u_given_equity)
print("E(U(Y)|X=debt)=", e_u_given_x_debt)
print("E(U(Y)|X=equity)=", e_u_given_x_equity)

int_model_x_debt = do(model2, {"X": "debt"})
infer_debt = VariableElimination(int_model_x_debt)
marginal_u_given_debt = infer_debt.query(variables=['U'])
expectation_u_given_debt = get_expectation(marginal_u_given_debt)
print("E(U(Y_{X=debt}))=", expectation_u_given_debt)
int_model_x_equity = do(model2, {"X": "equity"})
infer_equity = VariableElimination(int_model_x_equity)
marginal_u_given_equity = infer_equity.query(variables=['U'])
expectation_u_given_equity = get_expectation(marginal_u_given_equity)
print("E(U(Y_{X=equity}))=", expectation_u_given_equity)

## リスト12.8

In [ ]:
model = BayesianNetwork(
    [
        ('intent', 'AI prediction'),
        ('intent', 'choice'),
        ('AI prediction', 'box B'),
        ('choice', 'U'),
        ('box B', 'U'),
    ]
)

## リスト12.9

In [ ]:
cpd_intent = TabularCPD(
    'intent', 2, [[0.5], [.5]],
    state_names={'intent': ['B', 'both']}
)
print(cpd_intent)

cpd_choice = TabularCPD(
    'choice', 2, [[1, 0], [0, 1]],
    evidence=['intent'],
    evidence_card=[2],
    state_names={
        'choice': ['B', 'both'],
        'intent': ['B', 'both']
    }
)
print(cpd_choice)

## リスト12.10

In [ ]:
cpd_AI = TabularCPD(
    'AI prediction', 2, [[.95, 0.05], [.05, .95]],
    evidence=['intent'],
    evidence_card=[2],
    state_names={
        'AI prediction': ['B', 'both'],
        'intent': ['B', 'both']
    }
)
print(cpd_AI)

cpd_box_b_content = TabularCPD(
    'box B', 2, [[0, 1], [1, 0]],
    evidence=['AI prediction'],
    evidence_card=[2],
    state_names={
        'box B': [0, 1000000],
        'AI prediction': ['B', 'both']
    }
)
print(cpd_box_b_content)

## リスト12.11

In [ ]:
cpd_u = TabularCPD(
    'U', 4,
    [
        [1, 0, 0, 0],
        [0, 1, 0, 0],
        [0, 0, 1, 0],
        [0, 0, 0, 1],
    ],
    evidence=['box B', 'choice'],
    evidence_card=[2, 2],
    state_names={
        'U': [0, 1000, 1000000, 1001000],
        'box B': [0, 1000000],
        'choice': ['B', 'both']
    }
)
print(cpd_u)

model.add_cpds(cpd_intent, cpd_choice, cpd_AI)
model.add_cpds(cpd_box_b_content, cpd_u)

## リスト12.12

In [ ]:
int_model_x_both = do(model, {"choice": "both"})
infer_both = VariableElimination(int_model_x_both)
marginal_u_given_both = infer_both.query(
    variables=['U'], evidence={'intent': 'both'})
expectation_u_given_both = get_expectation(marginal_u_given_both)
print("E(U(Y_{choice=both}|intent=both))=", expectation_u_given_both)
int_model_x_box_B = do(model, {"choice": "B"})
infer_box_B = VariableElimination(int_model_x_box_B)
marginal_u_given_box_B = infer_box_B.query(
    variables=['U'], evidence={'intent': 'both'})
expectation_u_given_box_B = get_expectation(marginal_u_given_box_B)
print("E(U(Y_{choice=box B}|intent=both))=", expectation_u_given_box_B)
int_model_x_both = do(model, {"choice": "both"})
infer_both = VariableElimination(int_model_x_both)
marginal_u_given_both = infer_both.query(
    variables=['U'], evidence={'intent': 'B'})
expectation_u_given_both = get_expectation(marginal_u_given_both)
print("E(U(Y_{choice=both}|intent=B))=", expectation_u_given_both)
int_model_x_box_B = do(model, {"choice": "B"})
infer_box_B = VariableElimination(int_model_x_box_B)
marginal_u_given_box_B = infer_box_B.query(
    variables=['U'], evidence={'intent': 'B'})
expectation_u_given_box_B = get_expectation(marginal_u_given_box_B)
print("E(U(Y_{choice=box B}|intent=B))=", expectation_u_given_box_B)